In [1]:

# #! echo $PWD
! pip install ../utility_functions/
# #!pip install ~/fwiViz/utility_functions/

Processing /home/jovyan/fwiVis/utility_functions
  Preparing metadata (setup.py) ... done
  Created wheel for fwiVis: filename=fwiVis-0.1-py3-none-any.whl size=17875 sha256=385012bebc33bd73acdea7ddbfcae3665afdee70b83d1ffd371a78602b41d9fe
  Stored in directory: /tmp/pip-ephem-wheel-cache-igebzyoi/wheels/79/0f/3d/08c18473dd7e0fb915900e6b4f13b81f1fa84371f9bf24d864
Successfully built fwiVis


In [2]:
import fwiVis.fwiVis as fv
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
from math import cos, asin, sqrt
import re

import numpy as np
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
import os
import rioxarray as rio
import xarray as xr
import rasterio
import glob
from shapely.errors import ShapelyDeprecationWarning
from shapely.geometry import Point
import warnings
import folium
import datetime
import time
from folium import plugins
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
#import contextily as cx
from shapely.geometry import box
import sys
from datetime import datetime, timedelta
from itertools import chain
from bs4 import BeautifulSoup # I mamba installed bs4
import requests

from datetime import date

In [65]:



# def make_df_of_fires(year = "2023", path_region = "Quebec_PostHoc", custom_path = "/home/jovyan/fireatlast_nrt/fireatlas/data/FEDSoutput-v3/"):

#     diroutdata = custom_path
#     spath = os.path.join(diroutdata, path_region, str(year), "Largefire")
#     fnms = [f for f in os.listdir(spath)]
#     print(fnms)
#     #fnms = fnms.sort()
#     tmp_ids = pd.DataFrame(fnms, columns=["ids"])
#     #tmp_ids = tmp_ids[~tmp_ids.ids.str.contains(".")]
#     print(tmp_ids)
#     tmp_ids = tmp_ids.ids.unique()
#     print(f'{len(tmp_ids)} unique ID found')

#     print("Reading in IDS")
#     ### reading in the ids
#     fires = pd.DataFrame()
#     for n,i in enumerate(tmp_ids, start = 0):
#         try:
#             foo = fv.load_large_fire(i, year = year, path_region= path_region, layer = "perimeter",  s3_path = False, custom_path = custom_path)
#             foo["fireID"] = str(i)
#         except Exception as e:
#             print("Error at ID: ",i)
#             print(e)
#             continue
#         ## Extract the period between 
#         fires = pd.concat([fires, foo])
#     return (fires)
#         #print(fires)
#             #fr_pd = pd.DataFrame(fires, columns=["lat", "lon", "farea", "data_source"])
#         #fires.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/"+"20_days_fire_stats_only_718270-99999_" +min_t + max_t + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv")

In [66]:
# fires = make_df_of_fires()

In [3]:
## but wait, maybe Julia made a function for combined lf anyway


lf = gpd.read_file("/home/jovyan/fireatlast_nrt/fireatlas/data/FEDSoutput-v3/Quebec_V3/2023/CombinedLargefire/20230915PM/lf_perimeter.fgb")

In [4]:
len(lf.fireID.unique())

204

In [65]:
def get_largest_perimeter(df):
    fireID = df.fireID
    df = df[(np.round(df.farea, 4) == np.round(df.farea.max(), 4)) & (df.t == df.t.max())]
    if(len(df) == 0):
        print(f"Warning! dropping ID {fireID} because max t didn't match max farea.")
    return(df)

In [87]:
new_lf = lf.groupby("fireID").apply(get_largest_perimeter,  include_groups = True )

/tmp/ipykernel_1768/2721545350.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  new_lf = lf.groupby("fireID").apply(get_largest_perimeter,  include_groups = True )


In [89]:
#new_lf.explore()

In [ ]:
print(len(lf.fireID.unique()))
len(new_lf.fireID.unique())

204


204

In [83]:
new_lf = new_lf.to_crs("4326")
new_lf["lat_centroid"] = new_lf.geometry.centroid.y
new_lf["lon_centroid"] = new_lf.geometry.centroid.x

new_lf.columns

/tmp/ipykernel_1768/2034074402.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  new_lf["lat_centroid"] = new_lf.geometry.centroid.y
/tmp/ipykernel_1768/2034074402.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  new_lf["lon_centroid"] = new_lf.geometry.centroid.x


Index(['mergeid', 'ftype', 'n_pixels', 'n_newpixels', 'farea', 'fperim',
       'flinelen', 'duration', 'pixden', 'meanFRP', 't', 't_st', 't_ed',
       'fireID', 'isignition', 't_inactive', 'isactive', 'isdead',
       'mayreactivate', 'geom_counts', 'low_confidence_grouping', 'region',
       'primarykey', 'geometry', 'lat_centroid', 'lon_centroid'],
      dtype='object')

In [ ]:
new_lf.fireID = new_lf.fireID.astype("int")
new_lf.mergeid = new_lf.mergeid.astype("int")

In [83]:
lf.fireID = lf.fireID.astype("int")
lf.mergeid = lf.mergeid.astype("int")

In [85]:
import datetime
now = date.today()
new_lf.fireID = new_lf.fireID.astype('int')
new_lf.mergeid = new_lf.mergeid.astype('int')
new_lf.lon_centroid = new_lf.centroid.x
new_lf.lat_centroid = new_lf.centroid.y
new_lf = new_lf.drop(columns = "geometry")

new_lf.to_csv("~/fwiVis/notebooks/data/Quebec_v3_last_perimeters_as_of_" + str(now.year)+str(now.month)+str(now.day)+".csv")

/tmp/ipykernel_1768/210436966.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  new_lf.lon_centroid = new_lf.centroid.x
/tmp/ipykernel_1768/210436966.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  new_lf.lat_centroid = new_lf.centroid.y


In [22]:
# print(len(lf[~lf.fireID.isin(new_lf.fireID.unique())].fireID.unique()))
# weird_ids = lf[~lf.fireID.isin(new_lf.fireID.unique())].fireID.unique()
# print(weird_ids)

28
[ 244.  605.   15. 1324.  270. 3038. 1089. 1034.  248. 1335.  315.  322.
  326.  328.  331.  332.  230. 2366. 2717. 2108. 1625. 1088.  235. 1271.
 3261. 1761. 2560. 2517.]


In [ ]:
def listFD(url, ext=''):
    page = requests.get(url).text
    #print(page)
    soup = BeautifulSoup(page, 'html.parser')
    return [url + '/' + node.get('href') for node in soup.find_all('a') if node.get('href').endswith(ext)]

## https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecAllFires.Radius.25.km.247.biggestFires/
# https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecAllFires.Radius.25.km.216.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/chicletDataNoSmoothing/
# https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecFEDSv3.Radius.50.km.176.biggestFires/
# 'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecAllFires.Radius.25.km.247.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/chicletDataNoSmoothing/
def get_nccs_url(pattern, url = 'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecFEDSv3.Radius.50.km.176.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/MAX/chicletDataNoSmoothing/', ext = 'csv', pattern2 = "FWI.raw"):    

#url = 'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecAllFires.Radius.25.km.216.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/chicletDataNoSmoothing/'
#ext = 'csv'
    file_list = []
    for file in listFD(url, ext):
        file_list.append(file)

    try_pd = pd.DataFrame(file_list, columns= ["urls"])
    size = try_pd[try_pd.urls.str.contains(pattern)].urls.values.size
    if(pattern2 is not None):
        #print(f"Searching by second pattern {pattern2}")
        size = try_pd[(try_pd.urls.str.contains(pattern)) & (try_pd.urls.str.contains(pattern2))].urls.values.size
        #print(size)
    if(size == 0):
        print("No matches found to pattern. Returning None.")
        return(None)
    if(size >= 2):
        print("Multiple matches found:")
        print(try_pd[try_pd.urls.str.contains(pattern)].urls.values)
        raise ValueError()
    url = try_pd[try_pd.urls.str.contains(pattern)].urls.values[0]
    if(pattern2 is not None):
        url = try_pd[try_pd.urls.str.contains(pattern) & try_pd.urls.str.contains(pattern2)].urls.values[0]
    return(url)

def get_gridded_fwi(fireID):
    
    # Get the URL for the file
    fireID = str(fireID)
    #pattern = "FWI." + fireID
    pattern = "\." + fireID + "\_"
    url = get_nccs_url(pattern = pattern)
    
    if(url is not None):
        
        # Get the DF
        grid_FWI = pd.read_csv(url)
        # Change names
        grid_FWI = grid_FWI.rename(columns={'INITDATE': 't', 
                                 "0":"FWI",
                                 "1":"FWI_lead_1",
                                 "2":"FWI_lead_2",
                                 "3":"FWI_lead_3",
                                 "4":"FWI_lead_4",
                                 "5":"FWI_lead_5",
                                 "6":"FWI_lead_6",
                                 "7":"FWI_lead_7",
                                 "8":"FWI_lead_8"
                                })
        # Change dates
        grid_FWI.t = grid_FWI.t.astype("datetime64[ns]").dt.strftime('%Y-%m-%d 12:00:00')

        # return
        return(grid_FWI)
    else:
        return(None)

<>:49: SyntaxWarning: invalid escape sequence '\.'
<>:49: SyntaxWarning: invalid escape sequence '\_'
<>:49: SyntaxWarning: invalid escape sequence '\.'
<>:49: SyntaxWarning: invalid escape sequence '\_'
/tmp/ipykernel_1768/1404090192.py:49: SyntaxWarning: invalid escape sequence '\.'
  pattern = "\." + fireID + "\_"
/tmp/ipykernel_1768/1404090192.py:49: SyntaxWarning: invalid escape sequence '\_'
  pattern = "\." + fireID + "\_"


In [ ]:
# lf.to_csv("~/fwiVis/notebooks/data/Quebec_v3_perimeters_as_of_" + str(now.year)+str(now.month)+str(now.day)+".csv")

AttributeError: 'DataFrame' object has no attribute 'explore'

In [84]:
def merge_gridded_FWI_data(df, FWI_subset_t = True):
    fireID = df.fireID.iloc[0]
    fwi = get_gridded_fwi(str(fireID))
    if(fwi is None):
        print(f"Warning:{fireID} was not extracted sucessfully.")
        return(None)
    else:
        if(FWI_subset_t):
            fwi = fwi[fwi.t.astype("datetime64[ns]") >= df.t.astype("datetime64[ns]").min()]
            fwi = fwi[fwi.t.astype("datetime64[ns]") <= df.t.astype("datetime64[ns]").max()]
        merged = df.merge(fwi, on = ["t"], how = "outer")
        return(merged)


In [85]:
merged_lf = lf.groupby("fireID").apply(merge_gridded_FWI_data)

No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches found to pattern. Returning None.
No matches

/tmp/ipykernel_3177/1404090192.py:49: SyntaxWarning: invalid escape sequence '\.'
  pattern = "\." + fireID + "\_"
/tmp/ipykernel_3177/1404090192.py:49: SyntaxWarning: invalid escape sequence '\_'
  pattern = "\." + fireID + "\_"


ValueError: 